In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
 
# ======================================================
# STEP 1: Data qubits and stabilizers
# ======================================================
n_data = 9
#   q0 q1 q2
#   q3 q4 q5
#   q6 q7 q8
 
x_checks = [[0, 1, 3, 4], [2, 5], [3, 6], [4, 5, 7, 8]]   # z-type stabilizers
z_checks = [[0, 1], [1, 2, 4, 5], [3, 4, 6, 7], [7, 8]]   # x-type stabilizers
 
n_x, n_z = len(x_checks), len(z_checks)

In [26]:
# STEP 2: Registers
#   17 qubits = 9 data + 4 X-ancilla + 4 Z-ancilla
# ======================================================
def measure_x_check(qc, data, anc, Z_stabilizer, clbit):
    qc.reset(anc)                 # fresh ancilla
    qc.h(anc)
    for ele_x_check in Z_stabilizer:
        qc.cx(anc, data[ele_x_check])       # ancilla controls, data is target
    qc.h(anc)
    qc.measure(anc, clbit)        # store result in the array
 
 
def measure_z_check(qc, data, anc, X_stabilizer, clbit):
    qc.reset(anc)                 # fresh ancilla
    for ele_z_check in X_stabilizer:
        qc.cx(data[ele_z_check], anc)       # data controls, ancilla is target
    qc.measure(anc, clbit)        # store result in the array
 
 
# ======================================================
# STEP 3: One stabilizer-measurement round
# ======================================================
def build_circuit(error_type=None, error_qubit=None):
    data = QuantumRegister(n_data, "q")
    anc = QuantumRegister(1, "anc")
 
    # classical arrays: x-syndrome and z-syndrome for each round
    sx = [ClassicalRegister(n_x, f"x_round{r}") for r in range(2)]
    sz = [ClassicalRegister(n_z, f"z_round{r}") for r in range(2)]
 
    qc = QuantumCircuit(data, anc, sx[0], sz[0], sx[1], sz[1])
 
    def one_round(r):
        for i, support in enumerate(x_checks):
            measure_x_check(qc, data, anc[0], support, sx[r][i])
        for i, support in enumerate(z_checks):
            measure_z_check(qc, data, anc[0], support, sz[r][i])
        qc.barrier()
 
    one_round(0)

    # build_circuit(error_type="Z", error_qubit=10)
    
    if error_type is not None:
        {"X": qc.x, "Y": qc.y, "Z": qc.z}[error_type.upper()](data[error_qubit])
        qc.barrier()
 
    one_round(1)
    return qc

# print(qc.draw(output="text", fold=150))

In [42]:
# STEP 4: Full circuit with an injected error
# ======================================================
def run(error_type=None, error_qubit=None):
    qc = build_circuit(error_type, error_qubit)
    counts = AerSimulator().run(qc, shots=1).result().get_counts()
    bitstring = next(iter(counts))
 
    # Qiskit prints the LAST register first, and bit 0 on the RIGHT.
    # Reverse the register order, then reverse the bits inside each one.
    parts = [[int(b) for b in p[::-1]] for p in bitstring.split()[::-1]]
    x0, z0, x1, z1 = parts            # creation order of the registers
 
    x_syndrome = [a ^ b for a, b in zip(x0, x1)]   # change between rounds
    z_syndrome = [a ^ b for a, b in zip(z0, z1)]
    return x_syndrome, z_syndrome

def fired_stabilizers(x_syn, z_syn):
    """Return the X-checks and Z-checks that fired, with their data-qubit support."""
    fired_x = [(f"X{i}", x_checks[i]) for i, bit in enumerate(x_syn) if bit]
    fired_z = [(f"Z{i}", z_checks[i]) for i, bit in enumerate(z_syn) if bit]
    return fired_x, fired_z
 
if __name__ == "__main__":
    error_type, error_qubit = "z", 0
 
    qc = build_circuit(error_type, error_qubit)
    print(qc.draw(output="text", fold=150))
    qc.draw(output="mpl", fold=60, filename="surface_code_single_ancilla.png")
 
    x_syn, z_syn = run(error_type, error_qubit)
    # print(f"\nInjected error : {error_type} on q{error_qubit}")
    print("Final X-stabilizer errors(Z-error):", x_syn)
    print("Final Z-stabilizer errors(X-error):", z_syn)

    fired_x, fired_z = fired_stabilizers(x_syn, z_syn)

    print("\nX-checks that detected the error:")
    if fired_x:
        for name, support in fired_x:
            print(f"  {name} -> qubits {support}")
    else:
        print("  none")

    print("Z-checks that detected the error:")
    if fired_z:
        for name, support in fired_z:
            print(f"  {name} -> qubits {support}")
    else:
        print("  none")

                      ┌───┐                                                                                                                          »
       q_0: ──────────┤ X ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────»
                      └─┬─┘┌───┐                                                                                                                     »
       q_1: ────────────┼──┤ X ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────»
                        │  └─┬─┘                            ┌───┐                                                                                    »
       q_2: ────────────┼────┼──────────────────────────────┤ X ├────────────────────────────────────────────────────────────────────────────────────»
                        │    │  ┌───┐                       └─┬─┘                       ┌───┐ 

In [43]:
# ======================================================
# STEP 6: Locate the error from the syndrome arrays
# ======================================================
def decode_syndrome(x_syn, z_syn):
    """
    Locate the data-qubit error from the final syndrome arrays.
    X-check fired -> Z-type error, Z-check fired -> X-type error.
    """
    fired_x = [i for i, b in enumerate(x_syn) if b]
    fired_z = [i for i, b in enumerate(z_syn) if b]

    def locate(fired, checks):
        # qubits common to ALL fired checks ...
        if not fired:
            return set()
        cand = set.intersection(*[set(checks[i]) for i in fired])
        # ... and unique to them (absent from every check that did not fire)
        for i, s in enumerate(checks):
            if i not in fired:
                cand -= set(s)
        return cand

    def equivalent(cand, checks):
        # candidates are equivalent if every pair is itself a stabilizer
        stabs = [set(s) for s in checks]
        c = sorted(cand)
        return len(c) > 1 and all({a, b} in stabs for a in c for b in c if a < b)

    z_cand = locate(fired_x, x_checks)   # where a Z error could be
    x_cand = locate(fired_z, z_checks)   # where an X error could be

    def describe(kind, cand, checks_same_type):
        q = sorted(cand)
        if len(q) == 1:
            return f"{kind} error at data qubit q{q[0]}"
        text = f"{kind} error at data qubit q" + " or q".join(map(str, q))
        if equivalent(cand, checks_same_type):
            text += " (equivalent: they differ by a stabilizer, same correction works)"
        return text

    if not fired_x and not fired_z:
        return ["No error detected"]

    # Both arrays have fired checks: look for the qubit common to both groups
    if fired_x and fired_z:
        both = x_cand & z_cand
        if both:
            q = sorted(both)
            return [f"Y error at data qubit q{q[0]}" if len(q) == 1
                    else "Y error at data qubit q" + " or q".join(map(str, q))]
        out = []
        if z_cand: out.append(describe("Z", z_cand, z_checks))
        if x_cand: out.append(describe("X", x_cand, x_checks))
        return out or ["Syndrome does not match a single-qubit error"]

    # Only one array has fired checks: use the element unique to the fired set
    if fired_z:
        return [describe("X", x_cand, x_checks)] if x_cand else \
               ["Syndrome does not match a single-qubit error"]
    return [describe("Z", z_cand, z_checks)] if z_cand else \
           ["Syndrome does not match a single-qubit error"]

In [44]:
print("\nDecoded result:")
for line in decode_syndrome(x_syn, z_syn):
    print(" ", line)


Decoded result:
  Z error at data qubit q0 or q1 (equivalent: they differ by a stabilizer, same correction works)


In [36]:
if __name__ == "__main__":
    error_type, error_qubit = "X", 8
 
    qc = build_circuit(error_type, error_qubit)
    print(qc.draw(output="text", fold=150))
    qc.draw(output="mpl", fold=60, filename="surface_code_single_ancilla.png")
 
    x_syn, z_syn = run(error_type, error_qubit)
    # print(f"\nInjected error : {error_type} on q{error_qubit}")
    print("Final X-stabilizer errors(Z-error):", x_syn)
    print("Final Z-stabilizer errors(X-error):", z_syn)

    fired_x, fired_z = fired_stabilizers(x_syn, z_syn)

    print("\nX-checks that detected the error:")
    if fired_x:
        for name, support in fired_x:
            print(f"  {name} -> qubits {support}")
    else:
        print("  none")

    print("Z-checks that detected the error:")
    if fired_z:
        for name, support in fired_z:
            print(f"  {name} -> qubits {support}")
    else:
        print("  none")

                      ┌───┐                                                                                                                          »
       q_0: ──────────┤ X ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────»
                      └─┬─┘┌───┐                                                                                                                     »
       q_1: ────────────┼──┤ X ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────»
                        │  └─┬─┘                            ┌───┐                                                                                    »
       q_2: ────────────┼────┼──────────────────────────────┤ X ├────────────────────────────────────────────────────────────────────────────────────»
                        │    │  ┌───┐                       └─┬─┘                       ┌───┐ 